# Information-theoretic conditioning of tonal coarticulation: Taiwan Mandarin (COSPRO) vs Taiwan Southern Min (TAT-MOE)

Cross-language analysis behind the ICPhS 2027 submission. One row of the data = one within-word syllable pair in one
direction (`position` = anticipatory / carryover); the outcome is the target syllable's boundary F0 (`target_f0`,
ln Hz z-scored within speaker), and coarticulation is the slope of `target_f0` on the neighbouring syllable's
boundary F0 (`context_f0`).

**Model (`m_lang_pos`, lmer, cell 9)**

`target_f0 ~ context_f0 * informativity * context_informativity * (unigram + gain) * pos * lang`
`+ tone contrasts within language + duration + target_portion + context_portion + context_f0 * unit_pos * lang`
`+ (1 | onset_segment_l) + (1 | offset_segment_l) + (1 | speaker) + (1 + context_f0 + context_f0:unigram + context_f0:gain | tone_pair_l) + (1 | word_l)`

* `informativity` = the target tone's measure chosen with `INF_COL`; `context_informativity` = the same measure for the
  context tone. Scaled within language (`INF_SCALE = "within"`, 1 SD units).
* `unigram` = word surprisal (−log2 p(word), from the LM, context-free); `gain` = contextual gain
  (unigram − contextual surprisal). `pos`, `lang`: effects-coded ±0.5 (anticipatory −, carryover +; Mandarin −, Southern Min +).

**Measures used in the paper** (set `INF_COL` in the Settings cell and rerun the whole notebook):

| run | `INF_COL` | role |
|---|---|---|
| main | `informativity_full_context` | target-tone informativity given the adjacent syllable, the target's own segments and its position in the word |
| robustness | `functional_load_full_context` | entropy-based functional load in the same context cells |
| robustness | `informativity` | positional informativity (adjacent syllable + position, no target segments) |

Fits, tables and figures go to `fits/<INF_COL>/`, so runs with different measures never overwrite or reuse each other.

**Cells**

| cell | what | paper |
|---|---|---|
| Settings, Data, helpers, `both` | load and combine the two `processed_data.csv` files | |
| lmer fit | `m_lang_pos` (≈ 1–1.5 h on all data) | |
| informativity effects | marginal effects of target / context informativity on the slope, by language and position | results |
| informativity × predictability | unigram / gain × informativity | results (null) |
| target × context | target × context informativity interaction, simple effects, fitted slopes | results (null) |
| … on unigram / gain | same, for the unigram and gain effects | supplementary |
| brms fit | Bayesian version (400 rows per speaker, no word RE; ≈ 3.5 h with cmdstanr) | results |
| brms contrasts | the lmer contrasts above, from the posterior | results |
| paper figures | Figures 1–4 (`fig*_brms.png/pdf`) | figures |
| figure coefficients | estimates, 95% CrI, pd, Bayes factors for every line in Figures 1–4 | results / table |
| exploratory cells (marked) | matched-informativity comparison, dedicated model | discussion only |

Run order: top to bottom. The lmer cells (contrasts) must be run before the brms cells, which reuse their contrast
functions. Everything not shown in a contrast is held at 0 (= its mean).


In [ ]:
# install.packages(c("lme4", "lmerTest", "dplyr", "readr", "tidyr", "ggplot2"))
suppressPackageStartupMessages({
  library(lme4); library(lmerTest)
  library(dplyr); library(readr); library(tidyr); library(ggplot2)
})

## Settings

In [ ]:
DATA            <- Sys.getenv("TSM_DATA", path.expand("~/tsm_mfa/all_condenser/processed_data.csv"))   # tsm_pipeline/08_build_processed.py --prep <dir>
COSPRO_DATA     <- Sys.getenv("COSPRO_DATA", "../cospro_pipeline/out/processed_data.csv")   # cospro_pipeline/08_build_processed.py
KEEP_COL        <- "keep"
INF_COL         <- "informativity_full_context"   # main; robustness: "functional_load_full_context", "informativity" (see top)
EXCLUDE_OOV     <- FALSE                    # (Mandarin only; kept for symmetry)
MAX_PER_SPEAKER <- Inf
SEED            <- 2026
FIT_DIR         <- file.path("fits", INF_COL)   # one folder per measure: fits are never reused across measures
dir.create(FIT_DIR, recursive = TRUE, showWarnings = FALSE)
INF_CONTEXT     <- TRUE      # also model the CONTEXT tone's informativity (column context_<INF_COL>)
WORD_RE         <- "intercept"   # word-type random effect: "none" | "intercept" = (1 | word) | "slope" = (1 + context_f0 || word)
WSUF            <- c(none = "", intercept = "_wordint", slope = "_word")[[WORD_RE]]   # suffix for saved fits
add_word <- function(f, g = "word", type = WORD_RE) {
  re <- switch(type, none = NULL, intercept = paste0("(1 | ", g, ")"),
               slope = paste0("(1 + context_f0 || ", g, ")"))
  if (is.null(re)) return(f)
  as.formula(paste(paste(deparse(f, width.cutoff = 500L), collapse = " "), "+", re), env = environment(f))
}
add_terms <- function(f, terms) as.formula(paste(paste(deparse(f, width.cutoff = 500L), collapse = " "), "+", terms),
                                           env = environment(f))

## Data

In [ ]:
TONES <- c("1", "2", "3", "5", "7")
z <- function(x) as.numeric(scale(x))

prep_data <- function(raw, keep_col = KEEP_COL, inf_col = INF_COL, exclude_oov = EXCLUDE_OOV,
                      max_per_speaker = MAX_PER_SPEAKER) {
  d <- raw %>%
    filter(.data[[keep_col]]) %>%
    mutate(
      informativity_raw = .data[[inf_col]],
      context_informativity_raw = if (INF_CONTEXT) .data[[paste0("context_", inf_col)]] else 0,
      target_dur  = if_else(position == "carryover", dur2, dur1),
      position    = factor(position, levels = c("anticipatory", "carryover")),
      pos         = if_else(position == "carryover", 0.5, -0.5),       # effects coding
      medial      = as.numeric(target_pos_class == "medial"),         # target word-medial (control)
      target_tone = factor(as.character(target_tone), levels = TONES),
      tone_pair   = factor(paste0(tone1, "_", tone2, "_", position, "_", target_pos_class, if (INF_CONTEXT) paste0("_", context_pos_class) else "")),
      speaker = factor(speaker), item_id = factor(item_id), word = factor(word),
      onset_segment = factor(onset_segment), offset_segment = factor(offset_segment)
    ) %>%
    filter(!is.na(target_tone), is.finite(target_dur), target_dur > 0,
           is.finite(unigram_surprisal), is.finite(context_gain), is.finite(informativity_raw),
           is.finite(context_informativity_raw))
  if (is.finite(max_per_speaker)) {
    set.seed(SEED)
    d <- d %>% group_by(speaker) %>% slice_sample(n = max_per_speaker) %>% ungroup()
  }
  contrasts(d$target_tone) <- contr.sum(length(TONES))     # tone 7 = negative sum of the other levels
  d
}
scale_vars <- function(d) d %>% mutate(
  unigram         = z(unigram_surprisal),
  gain            = z(context_gain),
  surprisal       = z(word_surprisal),        # only for the comparison model
  informativity   = z(informativity_raw),
  # context informativity on the TARGET's scale, so the two slopes are comparable
  context_informativity = (context_informativity_raw - mean(informativity_raw)) / sd(informativity_raw),
  duration        = z(log(target_dur)),
  target_portion  = z(target_portion),
  context_portion = z(context_portion)
)

raw <- read_csv(DATA, show_col_types = FALSE, guess_max = 100000)
d <- scale_vars(prep_data(raw))

# scaling constants, to translate effects back to bits / raw units
scaling <- tibble(
  variable = c("unigram", "gain", "surprisal", "informativity", "duration"),
  label = c("unigram surprisal (bits)", "contextual gain (bits)", "contextual surprisal (bits)",
            "informativity", "log target duration (s)"),
  mean = c(mean(d$unigram_surprisal), mean(d$context_gain), mean(d$word_surprisal),
           mean(d$informativity_raw), mean(log(d$target_dur))),
  sd   = c(sd(d$unigram_surprisal),   sd(d$context_gain),   sd(d$word_surprisal),
           sd(d$informativity_raw),   sd(log(d$target_dur))))
print(scaling)

cat(nrow(d), "rows |", nlevels(d$speaker), "speakers |", nlevels(d$item_id), "items |",
    n_distinct(d$word), "word types |", nlevels(d$tone_pair), "tone pair x position cells\n")
d %>% count(site) %>% print()
d %>% count(position, target_tone) %>% pivot_wider(names_from = target_tone, values_from = n) %>% print()

In [ ]:
# ---- helper functions used by both the cross-language and the Southern Min-only models ----
tidy_coefs <- function(m) as.data.frame(coef(summary(m))) %>%
  tibble::rownames_to_column("term") %>%
  rename(estimate = Estimate, se = `Std. Error`, t = `t value`, p = `Pr(>|t|)`) %>%
  mutate(l95 = estimate - 1.96 * se, u95 = estimate + 1.96 * se) %>%
  select(term, estimate, se, l95, u95, df, t, p) %>% as_tibble()
fmt <- function(x) x %>% mutate(across(c(estimate, se, l95, u95, t), ~ round(.x, 3)),
                               df = round(df), p = signif(p, 3))
key <- c("context_f0", "context_f0:unigram", "context_f0:gain", "context_f0:surprisal",
         "context_f0:informativity", "context_f0:pos", "context_f0:unigram:pos",
         "context_f0:gain:pos", "context_f0:surprisal:pos", "context_f0:informativity:pos")
rob <- list()
add_rob <- function(name, mm) { rob[[name]] <<- tidy_coefs(mm) %>% mutate(n = nobs(mm)); invisible(NULL) }

slope_ci <- function(m, grid, data = d, factor_col = "target_tone") {
  tt <- delete.response(terms(m, fixed.only = TRUE))
  base <- grid
  if (!is.null(factor_col))
    base[[factor_col]] <- factor(levels(data[[factor_col]])[1], levels = levels(data[[factor_col]]))
  for (v in setdiff(all.vars(tt), c(names(base), "context_f0"))) base[[v]] <- 0   # other predictors at 0
  ca <- if (is.null(factor_col)) NULL else setNames(list(contrasts(data[[factor_col]])), factor_col)
  L  <- model.matrix(tt, mutate(base, context_f0 = 1), contrasts.arg = ca) -
        model.matrix(tt, mutate(base, context_f0 = 0), contrasts.arg = ca)   # slope = L %*% beta
  b  <- fixef(m); V <- as.matrix(vcov(m))
  L  <- L[, names(b), drop = FALSE]
  est <- as.numeric(L %*% b); se <- sqrt(rowSums((L %*% V) * L))
  grid %>% mutate(est = est, l95 = est - 1.96 * se, u95 = est + 1.96 * se)
}


In [ ]:
INF_SCALE <- "within"     # or "pooled"
TM_TONES  <- c("35", "21", "51", "55")

cos_raw <- read_csv(COSPRO_DATA, show_col_types = FALSE, guess_max = 100000)
cos <- cos_raw %>% filter(keep) %>%
  mutate(informativity_raw = .data[[INF_COL]],
         context_informativity_raw = if (INF_CONTEXT) .data[[paste0("context_", INF_COL)]] else 0,
         target_dur = if_else(position == "carryover", dur2, dur1),
         tp = if (grepl("sandhi", INF_COL)) sub("-", "_", surface_pair) else paste0(tone1, "_", tone2),
         unit_prop = phrase_prop) %>%
  filter(as.character(target_tone) %in% TM_TONES, is.finite(target_dur), target_dur > 0,
         is.finite(unigram_surprisal), is.finite(context_gain), is.finite(informativity_raw),
           is.finite(context_informativity_raw))
tsm <- prep_data(raw) %>% mutate(tp = paste0(tone1, "_", tone2), unit_prop = utt_prop)

cols <- c("target_f0", "context_f0", "position", "target_tone", "tp", "speaker", "item_id",
          "onset_segment", "offset_segment", "target_dur", "target_portion", "context_portion",
          "unigram_surprisal", "context_gain", "word_surprisal", "informativity_raw", "context_informativity_raw", "unit_prop", "word", "target_pos_class", "context_pos_class", "sex")
chr <- c("word", "target_tone", "speaker", "item_id", "onset_segment", "offset_segment", "position", "sex")
both <- bind_rows(
  cos %>% select(all_of(cols)) %>% mutate(across(all_of(chr), as.character), lang_lab = "TM"),
  tsm %>% select(all_of(cols)) %>% mutate(across(all_of(chr), as.character), lang_lab = "TSM")) %>%
  mutate(position = factor(position, levels = c("anticipatory", "carryover")),
         pos  = if_else(position == "carryover", 0.5, -0.5),
         lang = if_else(lang_lab == "TSM", 0.5, -0.5),
         tone_pair_l      = factor(paste(lang_lab, tp, position, target_pos_class, if (INF_CONTEXT) context_pos_class else "")),
         medial           = as.numeric(target_pos_class == "medial"),
         onset_segment_l  = factor(paste(lang_lab, onset_segment)),
         offset_segment_l = factor(paste(lang_lab, offset_segment)),
         speaker = factor(paste(lang_lab, speaker)),
         word_l  = factor(paste(lang_lab, word))) %>%
  group_by(lang_lab) %>%
  mutate(unigram = z(unigram_surprisal), gain = z(context_gain),
         duration = z(log(target_dur)), target_portion = z(target_portion),
         context_portion = z(context_portion), unit_pos = z(unit_prop),
         informativity = if (INF_SCALE == "within") z(informativity_raw) else informativity_raw,
         # context informativity on the target's scale
         context_informativity = if (INF_SCALE == "within") (context_informativity_raw - mean(informativity_raw)) / sd(informativity_raw)
                                 else context_informativity_raw) %>%
  ungroup()
if (INF_SCALE == "pooled") both <- both %>%
  mutate(context_informativity = (context_informativity - mean(informativity)) / sd(informativity),
         informativity = z(informativity))
# sum-coded target tone within each language, as numeric columns (0 for the other language)
tone_cols <- function(df, lab, levs) {
  cm <- contr.sum(length(levs)); rownames(cm) <- levs
  X <- matrix(0, nrow(df), ncol(cm))
  own <- df$lang_lab == lab
  X[own, ] <- cm[as.character(df$target_tone[own]), , drop = FALSE]
  colnames(X) <- paste0("tone_", lab, seq_len(ncol(X)))
  as_tibble(X)
}
both <- bind_cols(both, tone_cols(both, "TM", TM_TONES), tone_cols(both, "TSM", TONES))
TONE_TERMS <- paste(grep("^tone_(TM|TSM)[0-9]+$", names(both), value = TRUE), collapse = " + ")
stopifnot(!anyNA(select(both, matches("^tone_(TM|TSM)[0-9]+$"))))

both %>% group_by(lang_lab) %>%
  summarise(rows = n(), speakers = n_distinct(speaker), inf_mean = mean(informativity_raw),
            inf_sd = sd(informativity_raw), unigram_mean = mean(unigram_surprisal),
            gain_mean = mean(context_gain), r_gain_position = cor(gain, unit_pos), .groups = "drop") %>% print()
both <- both %>% mutate(sex_c = if_else(sex == "M", 0.5, -0.5))   # effects-coded: F = -0.5, M = +0.5
stopifnot(!anyNA(both$sex_c))

## lmer fit (`m_lang_pos`)

In [ ]:
f_lang <- as.formula(paste(
  "target_f0 ~ context_f0 * informativity * context_informativity * (unigram + gain) * pos * lang +", TONE_TERMS,
  "+ duration + target_portion + context_portion +",
  "(1 | onset_segment_l) + (1 | offset_segment_l) + (1 | speaker) + (1 + context_f0 + context_f0:unigram + context_f0:gain | tone_pair_l)"))
# f_lang <- add_terms(f_lang, "context_f0 * medial * lang")   # word-medial targets
f_lang <- add_word(f_lang, "word_l")   # WORD_RE (Settings)
# f_lang <- add_terms(f_lang, "context_f0 * informativity * (unigram + gain) * sex_c * lang")
f_lang_pos <- add_terms(f_lang, "context_f0 * unit_pos * lang")
f_lang_pos_txt <- paste(deparse(f_lang_pos, width.cutoff = 500L), collapse = " ")
print(f_lang_pos_txt)
# m_lang_pos supplies the formula and the contrast design for brms, and the lmer column of the
# comparison. Reuse a saved fit only if it was fitted with exactly this formula; otherwise refit
# (long: same model as the cross-language cell of analysis_lmer_tsm_citation.ipynb) and save it.
LMER_FILE <- file.path(FIT_DIR, paste0("m_lmer_lang_pos_for_brms_", INF_SCALE, WSUF, ".rds"))
m_lang_pos <- if (file.exists(LMER_FILE)) readRDS(LMER_FILE) else NULL
if (is.null(m_lang_pos) || !identical(attr(m_lang_pos, "f_txt"), f_lang_pos_txt)) {
  t0 <- Sys.time()
  m_lang_pos <- lmer(f_lang_pos, data = both, REML = TRUE, control = lmerControl(optimizer = "bobyqa"))
  cat("fitted m_lang_pos in", round(difftime(Sys.time(), t0, units = "mins"), 1), "min\n")
  attr(m_lang_pos, "f_txt") <- f_lang_pos_txt
  saveRDS(m_lang_pos, LMER_FILE)
} else cat("loaded", LMER_FILE, "\n")
tidy_coefs(m_lang_pos) %>% filter(grepl("context_f0", term)) %>% fmt() %>% print(n = Inf)

In [ ]:
# ===================== informativity (target / context) effects on the context-F0 slope =====================
# Marginal effect of X on the coarticulation slope = d²(target_f0) / d(context_f0) d(X), i.e. how much the slope of
# target F0 on context F0 changes per 1 SD of X (X = informativity or context_informativity), evaluated at
#   overall      : lang = 0 (average of the two languages, effects-coded ±0.5), pos = 0 (average of the positions)
#   per language : lang = -0.5 (Mandarin) / +0.5 (Southern Min), pos = 0
#   difference   : Southern Min - Mandarin, pos = 0
# All other continuous predictors at 0 (their means; unigram, gain, unit_pos, duration, portions, sex_c, medial);
# tone terms cancel. Tests: lmerTest::contest with Satterthwaite df.
# BY_POSITION = TRUE adds the anticipatory / carryover rows (x axis of the plots).
library(lmerTest)
MODEL       <- m_lang_pos
BY_POSITION <- TRUE
VARS   <- c(target = "informativity", context = "context_informativity")
HOLD   <- c("unigram", "gain", "informativity", "context_informativity", "duration", "target_portion",
            "context_portion", "unit_pos", "medial", "sex_c")
stopifnot(all(VARS %in% all.vars(formula(MODEL))))
if (!inherits(MODEL, "lmerModLmerTest")) MODEL <- as_lmerModLmerTest(MODEL)

tt_me  <- delete.response(terms(MODEL, fixed.only = TRUE))
ctr_me <- attr(model.matrix(MODEL), "contrasts")
bn_me  <- names(fixef(MODEL))
# contrast row: mixed difference over context_f0 (0/1) and var (0/1) at the given lang and pos
me_L <- function(var, lang_val, pos_val) {
  base <- both[1, ]
  for (v in intersect(HOLD, names(base))) base[[v]] <- 0
  base$lang <- lang_val; base$pos <- pos_val
  X <- function(cf0, x) { nd <- base; nd$context_f0 <- cf0; nd[[var]] <- x
                          model.matrix(tt_me, nd, contrasts.arg = ctr_me)[, bn_me, drop = FALSE] }
  X(1, 1) - X(1, 0) - X(0, 1) + X(0, 0)
}
poss <- if (BY_POSITION) c(average = 0, anticipatory = -0.5, carryover = 0.5) else c(average = 0)
rows <- list()
for (m in names(VARS)) for (p in names(poss)) {
  pv <- poss[[p]]
  rows[[length(rows) + 1]] <- list(measure = m, position = p, type = "overall",   language = "both languages", L = me_L(VARS[[m]], 0, pv))
  rows[[length(rows) + 1]] <- list(measure = m, position = p, type = "language",  language = "Mandarin",       L = me_L(VARS[[m]], -0.5, pv))
  rows[[length(rows) + 1]] <- list(measure = m, position = p, type = "language",  language = "Southern Min",   L = me_L(VARS[[m]], 0.5, pv))
  rows[[length(rows) + 1]] <- list(measure = m, position = p, type = "difference", language = "Southern Min - Mandarin",
                                   L = me_L(VARS[[m]], 0.5, pv) - me_L(VARS[[m]], -0.5, pv))
}
L_me <- do.call(rbind, lapply(rows, `[[`, "L"))
stopifnot(all(rowSums(abs(L_me)) > 0))

me_tab <- bind_cols(
  tibble(measure  = factor(sapply(rows, `[[`, "measure"), c("target", "context"),
                           c("target informativity", "context informativity")),
         position = factor(sapply(rows, `[[`, "position"), names(poss)),
         type     = sapply(rows, `[[`, "type"),
         language = factor(sapply(rows, `[[`, "language"), c("both languages", "Mandarin", "Southern Min", "Southern Min - Mandarin"))),
  contest(MODEL, L_me, joint = FALSE, ddf = "Satterthwaite", confint = TRUE) %>% as.data.frame() %>%
    transmute(estimate = Estimate, se = `Std. Error`, df, t = `t value`, l95 = lower, u95 = upper, p = `Pr(>|t|)`)) %>%
  mutate(across(c(estimate, se, t, l95, u95), ~ round(.x, 3)), df = round(df, 1), p = signif(p, 3))
print(as.data.frame(me_tab), row.names = FALSE)
write_csv(me_tab, file.path(FIT_DIR, paste0("informativity_target_context_effects_", INF_SCALE, WSUF, ".csv")))

# sanity check: the average-position difference rows equal the ...:lang coefficients
tidy_coefs(MODEL) %>% filter(term %in% c("context_f0:informativity:lang", "context_f0:lang:context_informativity",
                                         "context_f0:context_informativity:lang")) %>% fmt() %>% print()

# ---- Plots ----
me_plot <- function(tab, ttl, colour_by_language = FALSE) {
  aes_x <- if (BY_POSITION) aes(position, estimate) else aes(language, estimate)
  g <- ggplot(tab, aes_x) + geom_hline(yintercept = 0, linetype = 2, colour = "grey50")
  if (colour_by_language || BY_POSITION) {
    g <- g + geom_pointrange(aes(ymin = l95, ymax = u95, colour = language), position = position_dodge(width = 0.4))
  } else {
    g <- g + geom_pointrange(aes(ymin = l95, ymax = u95))
  }
  g + facet_wrap(~ measure) +
    labs(title = ttl, x = NULL, colour = NULL,
         y = "change in coarticulation slope\nper 1 SD informativity (95% CI)") +
    theme_minimal(base_size = 12) + theme(legend.position = "bottom")
}
p_me_overall <- me_plot(filter(me_tab, type == "overall"),    "Both languages")
p_me_lang    <- me_plot(filter(me_tab, type == "language"),   "By language", colour_by_language = TRUE)
p_me_diff    <- me_plot(filter(me_tab, type == "difference"), "Southern Min - Mandarin")
p_me_overall; p_me_lang; p_me_diff
for (nm in c("overall", "lang", "diff"))
  ggsave(file.path(FIT_DIR, paste0("informativity_target_context_", nm, "_", INF_SCALE, WSUF, ".png")),
         get(paste0("p_me_", nm)), width = 8, height = 4.5, dpi = 300)

In [ ]:
# ===================== informativity x predictability: effect on the unigram / gain effect on the context-F0 slope ======
# For PRED in {unigram, gain} and X in {informativity (target), context_informativity}:
#   d³(target_f0) / d(context_f0) d(PRED) d(X) = how much the PRED effect on the coarticulation slope changes per 1 SD
#   of X (equivalently: how much the X effect on the slope changes per 1 SD of PRED); i.e. context_f0:PRED:X, evaluated
#   overall (lang = 0), per language (lang = ±0.5) and as Southern Min - Mandarin, positions averaged (pos = 0) unless
#   BY_POSITION. Other continuous predictors at 0 (their means); tone terms cancel. lmerTest::contest, Satterthwaite df.
library(lmerTest)
MODEL       <- m_lang_pos
BY_POSITION <- TRUE
VARS  <- c(target = "informativity", context = "context_informativity")
HOLD  <- c("unigram", "gain", "informativity", "context_informativity", "duration", "target_portion",
           "context_portion", "unit_pos", "medial", "sex_c")
if (!inherits(MODEL, "lmerModLmerTest")) MODEL <- as_lmerModLmerTest(MODEL)
tt_ix  <- delete.response(terms(MODEL, fixed.only = TRUE))
ctr_ix <- attr(model.matrix(MODEL), "contrasts")
bn_ix  <- names(fixef(MODEL))

# mixed difference of target_f0 over context_f0 and `vars` (each 0/1), signs (-1)^(k - #ones)
ix_L <- function(vars, lang_val, pos_val) {
  base <- both[1, ]
  for (v in intersect(HOLD, names(base))) base[[v]] <- 0
  base$lang <- lang_val; base$pos <- pos_val
  corners <- expand.grid(rep(list(0:1), length(vars) + 1)); names(corners) <- c("context_f0", vars)
  L <- 0
  for (i in seq_len(nrow(corners))) {
    nd <- base; for (v in names(corners)) nd[[v]] <- corners[[v]][i]
    L <- L + (-1)^(ncol(corners) - sum(corners[i, ])) *
      model.matrix(tt_ix, nd, contrasts.arg = ctr_ix)[, bn_ix, drop = FALSE]
  }
  L
}

interaction_effects <- function(pred) {
  poss <- if (BY_POSITION) c(average = 0, anticipatory = -0.5, carryover = 0.5) else c(average = 0)
  rows <- list()
  for (m in names(VARS)) for (p in names(poss)) {
    v <- c(pred, VARS[[m]]); pv <- poss[[p]]
    rows <- c(rows, list(
      list(measure = m, position = p, type = "overall",    language = "both languages", L = ix_L(v, 0, pv)),
      list(measure = m, position = p, type = "language",   language = "Mandarin",       L = ix_L(v, -0.5, pv)),
      list(measure = m, position = p, type = "language",   language = "Southern Min",   L = ix_L(v, 0.5, pv)),
      list(measure = m, position = p, type = "difference", language = "Southern Min - Mandarin",
           L = ix_L(v, 0.5, pv) - ix_L(v, -0.5, pv))))
  }
  L <- do.call(rbind, lapply(rows, `[[`, "L"))
  miss <- rowSums(abs(L)) == 0
  if (any(miss)) stop("the model has no context_f0:", pred, ":<informativity> term for: ",
                      paste(unique(sapply(rows[miss], `[[`, "measure")), collapse = ", "))
  bind_cols(
    tibble(predictor = pred,
           measure  = factor(sapply(rows, `[[`, "measure"), c("target", "context"),
                             paste0(pred, " x ", c("target informativity", "context informativity"))),
           position = factor(sapply(rows, `[[`, "position"), names(poss)),
           type     = sapply(rows, `[[`, "type"),
           language = factor(sapply(rows, `[[`, "language"),
                             c("both languages", "Mandarin", "Southern Min", "Southern Min - Mandarin"))),
    contest(MODEL, L, joint = FALSE, ddf = "Satterthwaite", confint = TRUE) %>% as.data.frame() %>%
      transmute(estimate = Estimate, se = `Std. Error`, df, t = `t value`, l95 = lower, u95 = upper, p = `Pr(>|t|)`)) %>%
    mutate(across(c(estimate, se, t, l95, u95), ~ round(.x, 3)), df = round(df, 1), p = signif(p, 3))
}

ix_plot <- function(tab, ttl, colour_by_language = FALSE) {
  aes_x <- if (BY_POSITION) aes(position, estimate) else aes(language, estimate)
  g <- ggplot(tab, aes_x) + geom_hline(yintercept = 0, linetype = 2, colour = "grey50")
  g <- if (colour_by_language || BY_POSITION)
    g + geom_pointrange(aes(ymin = l95, ymax = u95, colour = language), position = position_dodge(width = 0.4))
  else g + geom_pointrange(aes(ymin = l95, ymax = u95))
  g + facet_wrap(~ measure) +
    labs(title = ttl, x = NULL, colour = NULL,
         y = paste0("change in the ", unique(tab$predictor), " effect on the slope\nper 1 SD informativity (95% CI)")) +
    theme_minimal(base_size = 12) + theme(legend.position = "bottom")
}

ix_tabs <- list(); ix_plots <- list()
for (pred in c("unigram", "gain")) {
  tab <- interaction_effects(pred)
  ix_tabs[[pred]] <- tab
  cat("\n=====", pred, "x informativity =====\n"); print(as.data.frame(select(tab, -predictor)), row.names = FALSE)
  ix_plots[[paste0(pred, "_overall")]] <- ix_plot(filter(tab, type == "overall"),    paste(pred, "x informativity: both languages"))
  ix_plots[[paste0(pred, "_lang")]]    <- ix_plot(filter(tab, type == "language"),   paste(pred, "x informativity: by language"), TRUE)
  ix_plots[[paste0(pred, "_diff")]]    <- ix_plot(filter(tab, type == "difference"), paste(pred, "x informativity: Southern Min - Mandarin"))
}
write_csv(bind_rows(ix_tabs), file.path(FIT_DIR, paste0("informativity_x_unigram_gain_effects_", INF_SCALE, WSUF, ".csv")))

# sanity check: the average-position difference rows equal these ...:lang coefficients
tidy_coefs(MODEL) %>% filter(grepl("^context_f0:", term), grepl("lang", term), grepl("informativity", term),
                             grepl("unigram|gain", term), !grepl("pos|unit_pos", term)) %>% fmt() %>% print()

for (nm in names(ix_plots)) {
  print(ix_plots[[nm]])
  ggsave(file.path(FIT_DIR, paste0("informativity_x_", nm, "_", INF_SCALE, WSUF, ".png")),
         ix_plots[[nm]], width = 8, height = 4.5, dpi = 300)
}

In [ ]:
# ===================== target x context informativity =====================
# (a) interaction   : d³(target_f0) / d(context_f0) d(informativity) d(context_informativity) = how much the TARGET-
#                     informativity effect on the coarticulation slope changes per 1 SD of CONTEXT informativity
#                     (= context_f0:informativity:context_informativity), overall / by language / Southern Min - Mandarin
# (b) simple effects: the target-informativity effect on the slope at low / mean / high context informativity
#                     (-1 / 0 / +1 SD), and the context-informativity effect at low / mean / high target informativity
# (c) fitted slopes : coarticulation slope along target informativity (-2..2 SD), one line per context-informativity
#                     level, by language x position, with 95% CIs
# Positions: anticipatory / carryover (POSS). Other continuous predictors at 0 (their means); tone terms cancel.
library(lmerTest)
MODEL <- m_lang_pos
POSS  <- c(anticipatory = -0.5, carryover = 0.5)       # add average = 0 for the position-averaged rows
LANGS <- c(Mandarin = -0.5, "Southern Min" = 0.5)
HOLD  <- c("unigram", "gain", "informativity", "context_informativity", "duration", "target_portion",
           "context_portion", "unit_pos", "medial", "sex_c")
if (!inherits(MODEL, "lmerModLmerTest")) MODEL <- as_lmerModLmerTest(MODEL)
stopifnot("the model has no context_f0 x informativity x context_informativity term" =
            any(grepl("context_f0", names(fixef(MODEL))) & grepl("informativity:.*context_informativity|context_informativity:.*informativity",
                                                                   names(fixef(MODEL)))))
tt_tc  <- delete.response(terms(MODEL, fixed.only = TRUE))
ctr_tc <- attr(model.matrix(MODEL), "contrasts")
bn_tc  <- names(fixef(MODEL))

# mixed difference over context_f0 and `vars` (0/1 corners) with other variables fixed at `at`
L_at <- function(vars, lang_val, pos_val, at = list()) {
  base <- both[1, ]
  for (v in intersect(HOLD, names(base))) base[[v]] <- 0
  base$lang <- lang_val; base$pos <- pos_val
  for (v in names(at)) base[[v]] <- at[[v]]
  corners <- expand.grid(rep(list(0:1), length(vars) + 1)); names(corners) <- c("context_f0", vars)
  L <- 0
  for (i in seq_len(nrow(corners))) {
    nd <- base; for (v in names(corners)) nd[[v]] <- corners[[v]][i]
    L <- L + (-1)^(ncol(corners) - sum(corners[i, ])) *
      model.matrix(tt_tc, nd, contrasts.arg = ctr_tc)[, bn_tc, drop = FALSE]
  }
  L
}
run_contest <- function(L, info) {
  bind_cols(info,
            contest(MODEL, L, joint = FALSE, ddf = "Satterthwaite", confint = TRUE) %>% as.data.frame() %>%
              transmute(estimate = Estimate, se = `Std. Error`, df, t = `t value`, l95 = lower, u95 = upper, p = `Pr(>|t|)`)) %>%
    mutate(across(c(estimate, se, t, l95, u95), ~ round(.x, 3)), df = round(df, 1), p = signif(p, 3))
}
lang_levels <- c("both languages", "Mandarin", "Southern Min", "Southern Min - Mandarin")

# ---- (a) the interaction ----
V2 <- c("informativity", "context_informativity")
info_a <- list(); L_a <- list()
for (p in names(POSS)) {
  pv <- POSS[[p]]
  for (lg in c("both languages", "Mandarin", "Southern Min", "Southern Min - Mandarin")) {
    L <- switch(lg, "both languages" = L_at(V2, 0, pv), "Mandarin" = L_at(V2, -0.5, pv),
                "Southern Min" = L_at(V2, 0.5, pv), "Southern Min - Mandarin" = L_at(V2, 0.5, pv) - L_at(V2, -0.5, pv))
    L_a[[length(L_a) + 1]] <- L
    info_a[[length(info_a) + 1]] <- tibble(position = p, language = lg,
                                           type = ifelse(lg == "both languages", "overall", ifelse(grepl(" - ", lg), "difference", "language")))
  }
}
tc_int <- run_contest(do.call(rbind, L_a), bind_rows(info_a)) %>%
  mutate(position = factor(position, names(POSS)), language = factor(language, lang_levels))
cat("\n===== (a) target x context informativity interaction (change in the target-informativity effect on the slope per 1 SD context informativity) =====\n")
print(as.data.frame(tc_int), row.names = FALSE)

# ---- (b) simple effects ----
info_b <- list(); L_b <- list()
for (p in names(POSS)) for (lg in names(LANGS)) for (lev in c(-1, 0, 1)) {
  L_b[[length(L_b) + 1]] <- L_at("informativity", LANGS[[lg]], POSS[[p]], list(context_informativity = lev))
  info_b[[length(info_b) + 1]] <- tibble(effect = "target informativity", at = "context informativity", level = lev, position = p, language = lg)
  L_b[[length(L_b) + 1]] <- L_at("context_informativity", LANGS[[lg]], POSS[[p]], list(informativity = lev))
  info_b[[length(info_b) + 1]] <- tibble(effect = "context informativity", at = "target informativity", level = lev, position = p, language = lg)
}
tc_simple <- run_contest(do.call(rbind, L_b), bind_rows(info_b)) %>%
  mutate(effect = factor(effect, c("target informativity", "context informativity")),
         position = factor(position, names(POSS)),
         level_lab = factor(level, c(-1, 0, 1), c("low (-1 SD)", "mean", "high (+1 SD)")))
cat("\n===== (b) simple effects on the slope (per 1 SD), at low / mean / high levels of the other informativity =====\n")
print(as.data.frame(select(tc_simple, -level_lab)), row.names = FALSE)

# ---- (c) fitted slopes ----
b_tc <- fixef(MODEL); V_tc <- as.matrix(vcov(MODEL))
grid_c <- expand_grid(language = names(LANGS), position = names(POSS), informativity = seq(-2, 2, by = 0.25),
                      context_informativity = c(-1, 0, 1))
L_c <- do.call(rbind, lapply(seq_len(nrow(grid_c)), function(i) with(grid_c[i, ],
          L_at(character(0), LANGS[[language]], POSS[[position]],
               list(informativity = informativity, context_informativity = context_informativity)))))
est <- as.numeric(L_c %*% b_tc); se <- sqrt(rowSums((L_c %*% V_tc) * L_c))
tc_slopes <- grid_c %>% mutate(est = est, l95 = est - 1.96 * se, u95 = est + 1.96 * se,
                               position = factor(position, names(POSS)),
                               context_lab = factor(context_informativity, c(-1, 0, 1), c("low (-1 SD)", "mean", "high (+1 SD)")))

write_csv(tc_int,    file.path(FIT_DIR, paste0("target_x_context_informativity_interaction_", INF_SCALE, WSUF, ".csv")))
write_csv(tc_simple, file.path(FIT_DIR, paste0("target_x_context_informativity_simple_", INF_SCALE, WSUF, ".csv")))

# sanity check: Southern Min - Mandarin, averaged over positions = the ...:lang coefficient
tidy_coefs(MODEL) %>% filter(grepl("^context_f0:", term), grepl("context_informativity", term),
                             grepl("(^|:)informativity(:|$)", term), grepl("lang", term), !grepl("pos|unigram|gain", term)) %>%
  fmt() %>% print()

# ---- Plots ----
dodge <- position_dodge(width = 0.4)
theme_tc <- theme_minimal(base_size = 12) + theme(legend.position = "bottom")
# 1. the interaction, by position: overall / by language / difference as three panels
p_tc_int <- ggplot(tc_int, aes(position, estimate, colour = language)) +
  geom_hline(yintercept = 0, linetype = 2, colour = "grey50") +
  geom_pointrange(aes(ymin = l95, ymax = u95), position = dodge) +
  facet_wrap(~ type, labeller = as_labeller(c(overall = "both languages", language = "by language",
                                              difference = "Southern Min - Mandarin"))) +
  labs(x = NULL, colour = NULL, title = "target x context informativity",
       y = "change in the target-informativity effect on the slope\nper 1 SD context informativity (95% CI)") + theme_tc
# 2. simple effects: each informativity's effect at low / mean / high levels of the other one
p_tc_simple <- ggplot(tc_simple, aes(level_lab, estimate, colour = language)) +
  geom_hline(yintercept = 0, linetype = 2, colour = "grey50") +
  geom_pointrange(aes(ymin = l95, ymax = u95), position = dodge) +
  facet_grid(position ~ effect, labeller = labeller(effect = function(x) paste(x, "effect"))) +
  labs(x = "level of the other informativity", colour = NULL,
       y = "change in coarticulation slope per 1 SD (95% CI)") + theme_tc
# 3. fitted coarticulation slopes
p_tc_slopes <- ggplot(tc_slopes, aes(informativity, est, colour = context_lab, fill = context_lab)) +
  geom_ribbon(aes(ymin = l95, ymax = u95), alpha = 0.12, colour = NA) +
  geom_line(linewidth = 0.9) +
  facet_grid(position ~ language) +
  labs(x = "target informativity (SD)", y = "coarticulation slope (95% CI)",
       colour = "context informativity", fill = "context informativity") + theme_tc
p_tc_int; p_tc_simple; p_tc_slopes
ggsave(file.path(FIT_DIR, paste0("target_x_context_interaction_", INF_SCALE, WSUF, ".png")), p_tc_int, width = 10, height = 4.5, dpi = 300, bg = "white")
ggsave(file.path(FIT_DIR, paste0("target_x_context_simple_", INF_SCALE, WSUF, ".png")), p_tc_simple, width = 9, height = 6.5, dpi = 300, bg = "white")
ggsave(file.path(FIT_DIR, paste0("target_x_context_slopes_", INF_SCALE, WSUF, ".png")), p_tc_slopes, width = 9, height = 6.5, dpi = 300, bg = "white")

In [ ]:
# ===================== target x context informativity: effects on the UNIGRAM / GAIN effect on the slope =============
# Same three analyses as the target x context informativity cell, but for the effect of informativity on
# context_f0:PRED (PRED = unigram or gain), i.e. on how much PRED changes the coarticulation slope:
# (a) interaction   : d⁴(target_f0) / d(context_f0) d(PRED) d(informativity) d(context_informativity)
#                     = context_f0:PRED:informativity:context_informativity (needs that term in the model; skipped if absent)
# (b) simple effects: d³ / d(context_f0) d(PRED) d(X) for X = target (context) informativity at low / mean / high
#                     context (target) informativity
# (c) fitted PRED effect on the slope, d² / d(context_f0) d(PRED), along target informativity, one line per
#                     context-informativity level, by language x position
# Needs the functions and settings of the target x context cell (L_at, run_contest, POSS, LANGS, MODEL, ...).
tc_pred <- function(pred) {
  out <- list()
  V2  <- c(pred, "informativity", "context_informativity")
  # ---- (a) ----
  info_a <- list(); L_a <- list()
  for (p in names(POSS)) {
    pv <- POSS[[p]]
    for (lg in lang_levels) {
      L <- switch(lg, "both languages" = L_at(V2, 0, pv), "Mandarin" = L_at(V2, -0.5, pv),
                  "Southern Min" = L_at(V2, 0.5, pv), "Southern Min - Mandarin" = L_at(V2, 0.5, pv) - L_at(V2, -0.5, pv))
      L_a[[length(L_a) + 1]] <- L
      info_a[[length(info_a) + 1]] <- tibble(position = p, language = lg,
                                             type = ifelse(lg == "both languages", "overall", ifelse(grepl(" - ", lg), "difference", "language")))
    }
  }
  L_a <- do.call(rbind, L_a)
  if (all(L_a == 0)) {
    cat("\n(a) skipped: the model has no context_f0:", pred, ":informativity:context_informativity term\n", sep = "")
    out$int <- NULL
  } else {
    out$int <- run_contest(L_a, bind_rows(info_a)) %>%
      mutate(position = factor(position, names(POSS)), language = factor(language, lang_levels))
    cat("\n=====", pred, "(a): change in the target-informativity effect on the", pred, "effect per 1 SD context informativity =====\n")
    print(as.data.frame(out$int), row.names = FALSE)
  }
  # ---- (b) ----
  info_b <- list(); L_b <- list()
  for (p in names(POSS)) for (lg in names(LANGS)) for (lev in c(-1, 0, 1)) {
    L_b[[length(L_b) + 1]] <- L_at(c(pred, "informativity"), LANGS[[lg]], POSS[[p]], list(context_informativity = lev))
    info_b[[length(info_b) + 1]] <- tibble(effect = "target informativity", level = lev, position = p, language = lg)
    L_b[[length(L_b) + 1]] <- L_at(c(pred, "context_informativity"), LANGS[[lg]], POSS[[p]], list(informativity = lev))
    info_b[[length(info_b) + 1]] <- tibble(effect = "context informativity", level = lev, position = p, language = lg)
  }
  out$simple <- run_contest(do.call(rbind, L_b), bind_rows(info_b)) %>%
    mutate(effect = factor(effect, c("target informativity", "context informativity")),
           position = factor(position, names(POSS)),
           level_lab = factor(level, c(-1, 0, 1), c("low (-1 SD)", "mean", "high (+1 SD)")))
  cat("\n=====", pred, "(b): change in the", pred, "effect on the slope per 1 SD informativity, at low / mean / high levels of the other =====\n")
  print(as.data.frame(select(out$simple, -level_lab)), row.names = FALSE)
  # ---- (c) ----
  L_c <- do.call(rbind, lapply(seq_len(nrow(grid_c)), function(i) with(grid_c[i, ],
            L_at(pred, LANGS[[language]], POSS[[position]],
                 list(informativity = informativity, context_informativity = context_informativity)))))
  est <- as.numeric(L_c %*% b_tc); se <- sqrt(rowSums((L_c %*% V_tc) * L_c))
  out$slopes <- grid_c %>% mutate(est = est, l95 = est - 1.96 * se, u95 = est + 1.96 * se,
                                  position = factor(position, names(POSS)),
                                  context_lab = factor(context_informativity, c(-1, 0, 1), c("low (-1 SD)", "mean", "high (+1 SD)")))
  if (!is.null(out$int)) write_csv(out$int, file.path(FIT_DIR, paste0("target_x_context_", pred, "_interaction_", INF_SCALE, WSUF, ".csv")))
  write_csv(out$simple, file.path(FIT_DIR, paste0("target_x_context_", pred, "_simple_", INF_SCALE, WSUF, ".csv")))

  # ---- plots ----
  if (!is.null(out$int)) {
    out$p_int <- ggplot(out$int, aes(position, estimate, colour = language)) +
      geom_hline(yintercept = 0, linetype = 2, colour = "grey50") +
      geom_pointrange(aes(ymin = l95, ymax = u95), position = dodge) +
      facet_wrap(~ type, labeller = as_labeller(c(overall = "both languages", language = "by language",
                                                  difference = "Southern Min - Mandarin"))) +
      labs(x = NULL, colour = NULL, title = paste("target x context informativity, on the", pred, "effect"),
           y = paste0("change in the target-informativity effect on the\n", pred,
                      " effect per 1 SD context informativity (95% CI)")) + theme_tc
  }
  out$p_simple <- ggplot(out$simple, aes(level_lab, estimate, colour = language)) +
    geom_hline(yintercept = 0, linetype = 2, colour = "grey50") +
    geom_pointrange(aes(ymin = l95, ymax = u95), position = dodge) +
    facet_grid(position ~ effect, labeller = labeller(effect = function(x) paste(x, "effect"))) +
    labs(x = "level of the other informativity", colour = NULL, title = paste("effects on the", pred, "effect"),
         y = paste0("change in the ", pred, " effect on the slope\nper 1 SD informativity (95% CI)")) + theme_tc
  out$p_slopes <- ggplot(out$slopes, aes(informativity, est, colour = context_lab, fill = context_lab)) +
    geom_hline(yintercept = 0, linetype = 2, colour = "grey50") +
    geom_ribbon(aes(ymin = l95, ymax = u95), alpha = 0.12, colour = NA) +
    geom_line(linewidth = 0.9) +
    facet_grid(position ~ language) +
    labs(x = "target informativity (SD)", y = paste0(pred, " effect on the coarticulation slope\n(per 1 SD ", pred, ", 95% CI)"),
         colour = "context informativity", fill = "context informativity") + theme_tc
  for (nm in c("int", "simple", "slopes")) {
    g <- out[[paste0("p_", nm)]]
    if (is.null(g)) next
    print(g)
    ggsave(file.path(FIT_DIR, paste0("target_x_context_", pred, "_", nm, "_", INF_SCALE, WSUF, ".png")), g,
           width = if (nm == "int") 10 else 9, height = if (nm == "int") 4.5 else 6.5, dpi = 300, bg = "white")
  }
  invisible(out)
}
tc_unigram <- tc_pred("unigram")
tc_gain    <- tc_pred("gain")

## brms fit and posterior contrasts

In [ ]:
# ===================== Bayesian version of the current cross-language model (m_lang_pos, brms) =====================
# Same fixed and random structure as m_lang_pos: context_f0 * informativity * (unigram + gain) * pos * lang,
# medial and unit_pos controls, (1 + context_f0 + context_f0:unigram | tone_pair_l), word_l if WORD_RE.
# Reports the same three effects as the contest() cell (unigram, informativity, unigram x informativity),
# as posterior contrasts, side by side with lmer. Run once with DEV = TRUE, check brms ~ lmer, then DEV = FALSE.
# The fit is cached in FIT_DIR (file_refit = "on_change").
suppressPackageStartupMessages({ library(brms); library(posterior) })
options(mc.cores = parallel::detectCores())
BACKEND <- if (requireNamespace("cmdstanr", quietly = TRUE)) "cmdstanr" else "rstan"

DEV               <- FALSE  # TRUE: quick check (100 rows/speaker, 1000 iterations); FALSE: the real fit
MAX_PER_SPEAKER_B <- 400    # rows per speaker in the real fit (effects are tested against ~100 tone-pair cells)
B_WORD_RE         <- FALSE   # keep the word random effect of m_lang_pos (slow: ~20k word levels);
                            # FALSE drops it for the brms fit only (much faster; lmer shows it barely moves these contrasts)
SEED              <- 2026

dat_b <- both
n_cap <- if (DEV) 100 else MAX_PER_SPEAKER_B
if (is.finite(n_cap)) {
  set.seed(SEED)
  dat_b <- dat_b %>% group_by(speaker) %>% slice_sample(n = n_cap) %>% ungroup()
}

# formula of m_lang_pos, rebuilt from its fixed part and its random-effect terms, so the word terms can be dropped
f_lme <- formula(m_lang_pos)
bars  <- lme4::findbars(f_lme)                                   # one entry per random-effect term (|| already split)
if (!B_WORD_RE) bars <- Filter(function(b) deparse(b[[3]]) != "word_l", bars)
f_txt <- paste(paste(deparse(lme4::nobars(f_lme), width.cutoff = 500L), collapse = " "),
               paste0("(", sapply(bars, function(b) paste(deparse(b, width.cutoff = 500L), collapse = " ")), ")",
                      collapse = " + "),
               sep = " + ")
f_b <- bf(as.formula(f_txt))
cat(nrow(dat_b), "rows |", n_distinct(dat_b$word_l), "words | backend:", BACKEND, "| DEV =", DEV,
    "| word RE:", grepl("word_l", f_txt), "\n")

priors_b <- c(
  prior(normal(0, 1),   class = "Intercept"),
  prior(normal(0, 0.5), class = "b"),
  prior(exponential(2), class = "sd"),
  prior(exponential(2), class = "sigma"),
  prior(lkj(2),         class = "cor"))

t0 <- Sys.time()
fit_b <- brm(
  f_b, data = dat_b, family = gaussian(), prior = priors_b,
  chains = 4, iter = if (DEV) 1000 else 2000, warmup = if (DEV) 500 else 1000,
  cores = 4, seed = SEED, backend = BACKEND,
  threads = if (BACKEND == "cmdstanr" && !DEV) threading(2) else NULL,
  control = list(adapt_delta = 0.95),
  file = file.path(FIT_DIR, paste0("brm_lang_pos_v2_", INF_SCALE, if (grepl("word_l", f_txt)) "_word" else "",
                                   if (DEV) "_dev" else paste0("_cap", n_cap))),
  file_refit = "on_change")
cat("fitted in", round(difftime(Sys.time(), t0, units = "mins"), 1), "min\n")

diag_b <- summarise_draws(as_draws_df(fit_b, variable = "^b_|^sd_|^sigma", regex = TRUE),
                          "rhat", "ess_bulk", "ess_tail")
cat(sprintf("max Rhat %.3f | min bulk ESS %.0f | min tail ESS %.0f | divergences %d\n",
            max(diag_b$rhat, na.rm = TRUE), min(diag_b$ess_bulk, na.rm = TRUE),
            min(diag_b$ess_tail, na.rm = TRUE),
            sum(nuts_params(fit_b)$Value[nuts_params(fit_b)$Parameter == "divergent__"])))

# ---- posterior contrasts: same rows as the contest() cell ----
tt_b    <- delete.response(terms(m_lang_pos, fixed.only = TRUE))
ctr_b   <- attr(model.matrix(m_lang_pos), "contrasts")
lme_nms <- names(fixef(m_lang_pos))
brm_nms <- paste0("b_", ifelse(lme_nms == "(Intercept)", "Intercept", lme_nms))
stopifnot(all(brm_nms %in% variables(fit_b)))

HOLD_B <- c("unigram", "gain", "informativity", "duration", "target_portion", "context_portion", "unit_pos", "medial", "sex_c", "context_informativity")
slope_L_b <- function(vars, lang_val, pos_val) {        # mixed difference over context_f0 and vars
  base <- both[1, ]; base$lang <- lang_val; base$pos <- pos_val
  for (v in intersect(setdiff(HOLD_B, vars), names(base))) base[[v]] <- 0
  corners <- expand.grid(rep(list(0:1), length(vars) + 1)); names(corners) <- c("context_f0", vars)
  L <- 0
  for (i in seq_len(nrow(corners))) {
    nd <- base; for (v in names(corners)) nd[[v]] <- corners[[v]][i]
    L <- L + (-1)^(ncol(corners) - sum(corners[i, ])) *
      model.matrix(tt_b, nd, contrasts.arg = ctr_b)[, lme_nms, drop = FALSE]
  }
  L
}
LANG_LABELS <- c("-0.5" = "Mandarin", "0.5" = "Southern Min")
langs <- c(-0.5, 0.5)
poss  <- c(anticipatory = -0.5, average = 0, carryover = 0.5)
L_effect_b <- function(vars) {
  Lm <- do.call(rbind, lapply(langs, function(l) do.call(rbind, lapply(poss, function(p) slope_L_b(vars, l, p)))))
  rownames(Lm) <- paste(rep(LANG_LABELS[as.character(langs)], each = length(poss)), names(poss), sep = " | ")
  Ld <- do.call(rbind, lapply(poss, function(p) slope_L_b(vars, langs[2], p) - slope_L_b(vars, langs[1], p)))
  rownames(Ld) <- paste0("Southern Min - Mandarin | ", names(poss))
  L <- rbind(Lm, Ld); stopifnot(any(L != 0)); L
}
EFFECTS <- list("unigram", "informativity", c("unigram", "informativity"))
if (INF_CONTEXT && any(grepl("context_informativity", lme_nms))) EFFECTS <- c(EFFECTS, list("context_informativity"))

B <- unclass(as_draws_matrix(fit_b, variable = brm_nms))[, brm_nms, drop = FALSE]   # draws x coefficients
eff_b <- bind_rows(lapply(EFFECTS, function(vars) {
  L <- L_effect_b(vars)
  post <- B %*% t(L)                                                                  # draws x contrasts
  tibble(effect = paste(vars, collapse = " x "), contrast = rownames(L),
         lmer = as.numeric(L %*% fixef(m_lang_pos)),
         estimate = colMeans(post), sd = apply(post, 2, sd),
         l95 = apply(post, 2, quantile, 0.025), u95 = apply(post, 2, quantile, 0.975),
         l80 = apply(post, 2, quantile, 0.10),  u80 = apply(post, 2, quantile, 0.90),
         p_neg = colMeans(post < 0))
})) %>%
  mutate(pd = pmax(p_neg, 1 - p_neg),                   # probability of direction
         across(c(lmer, estimate, sd, l95, u95, l80, u80), ~ round(.x, 3)), across(c(p_neg, pd), ~ round(.x, 3)))
print(as.data.frame(eff_b), row.names = FALSE)
write_csv(eff_b, file.path(FIT_DIR, paste0("effects_lang_pos_v2_", INF_SCALE, if (DEV) "_dev" else "", "_brms.csv")))

# ---- Plot: the three effects, posterior mean with 80% / 95% CrI ----
dodge <- position_dodge(width = 0.4)
p_eff_b <- eff_b %>%
  filter(!grepl(" - ", contrast), !grepl("average", contrast)) %>%
  tidyr::separate(contrast, into = c("lang", "position"), sep = " \\| ") %>%
  mutate(effect = factor(effect, c("unigram", "informativity", "unigram x informativity", "context_informativity"),
                         c("unigram (per 1 SD)", "informativity (per 1 SD)", "unigram x informativity",
                           "context informativity (per 1 SD of target informativity)"))) %>%
  ggplot(aes(position, estimate, colour = lang)) +
  geom_hline(yintercept = 0, linetype = 2, colour = "grey50") +
  geom_linerange(aes(ymin = l95, ymax = u95), position = dodge) +
  geom_linerange(aes(ymin = l80, ymax = u80), linewidth = 1.6, position = dodge) +
  geom_point(size = 2.5, position = dodge) +
  facet_wrap(~ effect) +
  labs(x = NULL, colour = NULL, y = "change in coarticulation slope\n(posterior mean, 80% / 95% CrI)") +
  theme_minimal(base_size = 12) + theme(legend.position = "bottom")
p_eff_b


### Exploratory: language difference at matched raw informativity (reported only as a tendency)

In [ ]:
# Southern Min vs Mandarin: unigram x target informativity (carryover) at a COMMON raw context informativity,
# per 1 pooled SD of target informativity (same question as the pooled model's -0.118 row)
st   <- both %>% group_by(lang_lab) %>% summarise(m = mean(informativity_raw), s = sd(informativity_raw))
m    <- setNames(st$m, st$lang_lab); s <- setNames(st$s, st$lang_lab)
sd_p <- sd(both$informativity_raw)
common_raw <- function(C_RAW) {
  L_lang <- function(lg, lv) L_at(c("unigram", "informativity"), lv, 0.5,
                                  list(context_informativity = (C_RAW - m[[lg]]) / s[[lg]])) * sd_p / s[[lg]]
  L <- rbind(L_lang("TM", -0.5), L_lang("TSM", 0.5), L_lang("TSM", 0.5) - L_lang("TM", -0.5))
  run_contest(L, tibble(context_raw = C_RAW, row = c("Mandarin", "Southern Min", "Southern Min - Mandarin")))
}
bind_rows(lapply(c(mean(both$informativity_raw), 0.08, 0.06), common_raw)) %>% print(n = Inf)

In [ ]:
# ===================== brms versions of the informativity analyses (lmer cells 10–13) =====================
# Same contrast rows as the lmer cells (me_L, ix_L, L_at), applied to the posterior draws of fit_b.
# Columns: lmer = lmer estimate of the same contrast; estimate/sd = posterior mean/SD; 80% and 95% CrI;
# pd = probability of direction; credible = "*" if the 95% CrI excludes 0.
# Needs fit_b (brms cell) and the lmer cells 10–12 run in this session (me_L, ix_L, L_at, POSS, LANGS, grid_c, ...).
suppressPackageStartupMessages(library(posterior))
stopifnot(exists("fit_b"), exists("me_L"), exists("ix_L"), exists("L_at"), exists("grid_c"), exists("lang_levels"))

lme_nms <- names(fixef(m_lang_pos))
brm_nms <- paste0("b_", ifelse(lme_nms == "(Intercept)", "Intercept", lme_nms))
stopifnot(all(brm_nms %in% variables(fit_b)))
B    <- unclass(as_draws_matrix(fit_b, variable = brm_nms))[, brm_nms, drop = FALSE]   # draws x coefficients
BSUF <- paste0("_", INF_SCALE, WSUF, "_brms")
POSS3 <- c(average = 0, anticipatory = -0.5, carryover = 0.5)
VARS  <- c(target = "informativity", context = "context_informativity")
MEAS  <- c(target = "target informativity", context = "context informativity")

# posterior summary of contrast rows L (one row per contrast)
post_summ <- function(L, info) {
  L <- as.matrix(L); stopifnot(ncol(L) == ncol(B))
  post <- B %*% t(L)
  bind_cols(info, tibble(
    lmer     = as.numeric(L %*% fixef(m_lang_pos)),
    estimate = colMeans(post), sd = apply(post, 2, sd),
    l95 = apply(post, 2, quantile, 0.025), u95 = apply(post, 2, quantile, 0.975),
    l80 = apply(post, 2, quantile, 0.10),  u80 = apply(post, 2, quantile, 0.90),
    pd  = pmax(colMeans(post < 0), colMeans(post > 0)))) %>%
    mutate(credible = ifelse(l95 > 0 | u95 < 0, "*", ""),
           across(c(lmer, estimate, sd, l95, u95, l80, u80), ~ round(.x, 3)), pd = round(pd, 3))
}
# overall / Mandarin / Southern Min / difference rows for each position; Lfun(lang_val, pos_val) -> contrast row
build_rows <- function(Lfun, extra = list(), poss = POSS3) {
  Ls <- list(); info <- list()
  for (p in names(poss)) {
    pv <- poss[[p]]
    Ls <- c(Ls, list(Lfun(0, pv), Lfun(-0.5, pv), Lfun(0.5, pv), Lfun(0.5, pv) - Lfun(-0.5, pv)))
    info <- c(info, list(tibble(position = p, type = c("overall", "language", "language", "difference"),
                                language = lang_levels)))
  }
  list(L = do.call(rbind, Ls), info = mutate(bind_rows(info), !!!extra))
}
post_geoms <- function(w = 0.4) {
  d <- position_dodge(width = w)
  list(geom_hline(yintercept = 0, linetype = 2, colour = "grey50"),
       geom_linerange(aes(ymin = l95, ymax = u95), position = d),
       geom_linerange(aes(ymin = l80, ymax = u80), linewidth = 1.6, position = d),
       geom_point(size = 2.3, position = d))
}
theme_b <- theme_minimal(base_size = 12) + theme(legend.position = "bottom")
save_b  <- function(g, nm, w = 8, h = 4.5) { print(g)
  ggsave(file.path(FIT_DIR, paste0(nm, BSUF, ".png")), g, width = w, height = h, dpi = 300, bg = "white") }
eff_plot_b <- function(tab, ttl, ylab) {
  ggplot(tab, aes(position, estimate, colour = language)) + post_geoms() + facet_wrap(~ measure) +
    labs(title = ttl, x = NULL, colour = NULL, y = ylab) + theme_b
}

# ======== 1. target / context informativity effects on the slope (= cell 10) ========
parts <- lapply(names(VARS), function(m)
  build_rows(function(l, p) me_L(VARS[[m]], l, p), list(measure = MEAS[[m]])))
me_b <- post_summ(do.call(rbind, lapply(parts, `[[`, "L")), bind_rows(lapply(parts, `[[`, "info"))) %>%
  mutate(measure = factor(measure, MEAS), position = factor(position, names(POSS3)),
         language = factor(language, lang_levels))
cat("\n===== 1. informativity effects on the coarticulation slope (brms) =====\n")
print(as.data.frame(me_b), row.names = FALSE)
write_csv(me_b, file.path(FIT_DIR, paste0("informativity_target_context_effects", BSUF, ".csv")))
ylab1 <- "change in coarticulation slope per 1 SD\n(posterior mean, 80% / 95% CrI)"
save_b(eff_plot_b(filter(me_b, type == "overall"),    "Both languages (brms)", ylab1), "informativity_target_context_overall")
save_b(eff_plot_b(filter(me_b, type == "language"),   "By language (brms)", ylab1),    "informativity_target_context_lang")
save_b(eff_plot_b(filter(me_b, type == "difference"), "Southern Min - Mandarin (brms)", ylab1), "informativity_target_context_diff")

# ======== 2. unigram / gain x informativity (= cell 11) ========
ix_b <- list()
for (pred in c("unigram", "gain")) {
  parts <- lapply(names(VARS), function(m)
    build_rows(function(l, p) ix_L(c(pred, VARS[[m]]), l, p),
               list(predictor = pred, measure = paste(pred, "x", MEAS[[m]]))))
  tab <- post_summ(do.call(rbind, lapply(parts, `[[`, "L")), bind_rows(lapply(parts, `[[`, "info"))) %>%
    mutate(measure = factor(measure, paste(pred, "x", MEAS)), position = factor(position, names(POSS3)),
           language = factor(language, lang_levels))
  ix_b[[pred]] <- tab
  cat("\n===== 2.", pred, "x informativity (brms) =====\n"); print(as.data.frame(select(tab, -predictor)), row.names = FALSE)
  ylab2 <- paste0("change in the ", pred, " effect on the slope\nper 1 SD informativity (80% / 95% CrI)")
  save_b(eff_plot_b(filter(tab, type == "overall"),    paste(pred, "x informativity: both languages (brms)"), ylab2),
         paste0("informativity_x_", pred, "_overall"))
  save_b(eff_plot_b(filter(tab, type == "language"),   paste(pred, "x informativity: by language (brms)"), ylab2),
         paste0("informativity_x_", pred, "_lang"))
  save_b(eff_plot_b(filter(tab, type == "difference"), paste(pred, "x informativity: Southern Min - Mandarin (brms)"), ylab2),
         paste0("informativity_x_", pred, "_diff"))
}
write_csv(bind_rows(ix_b), file.path(FIT_DIR, paste0("informativity_x_unigram_gain_effects", BSUF, ".csv")))

# ======== 3. target x context informativity: on the slope (= cell 12) and on the unigram / gain effect (= cell 13) ====
# pred = NULL: effects on the coarticulation slope; pred = "unigram"/"gain": effects on context_f0:pred
tc_block_b <- function(pred = NULL) {
  pv  <- if (is.null(pred)) character(0) else pred
  tag <- if (is.null(pred)) "" else paste0(pred, "_")
  on  <- if (is.null(pred)) "the slope" else paste("the", pred, "effect")
  out <- list()
  # (a) interaction
  a <- build_rows(function(l, p) L_at(c(pv, "informativity", "context_informativity"), l, p), poss = POSS)
  if (all(a$L == 0)) {
    cat("\n(a) skipped: no context_f0:", paste(c(pv, "informativity:context_informativity"), collapse = ":"), " term\n", sep = "")
  } else {
    out$int <- post_summ(a$L, a$info) %>%
      mutate(position = factor(position, names(POSS)), language = factor(language, lang_levels))
    cat("\n===== 3a. target x context informativity, on", on, "(brms) =====\n")
    print(as.data.frame(out$int), row.names = FALSE)
    write_csv(out$int, file.path(FIT_DIR, paste0("target_x_context_", tag, "interaction", BSUF, ".csv")))
    g <- ggplot(out$int, aes(position, estimate, colour = language)) + post_geoms() +
      facet_wrap(~ type, labeller = as_labeller(c(overall = "both languages", language = "by language",
                                                  difference = "Southern Min - Mandarin"))) +
      labs(x = NULL, colour = NULL, title = paste("target x context informativity, on", on, "(brms)"),
           y = paste0("change in the target-informativity effect on ", on,
                      "\nper 1 SD context informativity (80% / 95% CrI)")) + theme_b
    save_b(g, paste0("target_x_context_", tag, "int"), w = 10)
  }
  # (b) simple effects at low / mean / high levels of the other informativity
  L_b <- list(); info_b <- list()
  for (p in names(POSS)) for (lg in names(LANGS)) for (lev in c(-1, 0, 1)) {
    L_b <- c(L_b, list(L_at(c(pv, "informativity"), LANGS[[lg]], POSS[[p]], list(context_informativity = lev)),
                       L_at(c(pv, "context_informativity"), LANGS[[lg]], POSS[[p]], list(informativity = lev))))
    info_b <- c(info_b, list(tibble(effect = c("target informativity", "context informativity"),
                                    level = lev, position = p, language = lg)))
  }
  out$simple <- post_summ(do.call(rbind, L_b), bind_rows(info_b)) %>%
    mutate(effect = factor(effect, c("target informativity", "context informativity")),
           position = factor(position, names(POSS)),
           level_lab = factor(level, c(-1, 0, 1), c("low (-1 SD)", "mean", "high (+1 SD)")))
  cat("\n===== 3b. simple effects on", on, "per 1 SD, at low / mean / high levels of the other informativity (brms) =====\n")
  print(as.data.frame(select(out$simple, -level_lab)), row.names = FALSE)
  write_csv(out$simple, file.path(FIT_DIR, paste0("target_x_context_", tag, "simple", BSUF, ".csv")))
  g <- ggplot(out$simple, aes(level_lab, estimate, colour = language)) + post_geoms() +
    facet_grid(position ~ effect, labeller = labeller(effect = function(x) paste(x, "effect"))) +
    labs(x = "level of the other informativity", colour = NULL, title = paste("effects on", on, "(brms)"),
         y = paste0("change in ", on, " per 1 SD\n(80% / 95% CrI)")) + theme_b
  save_b(g, paste0("target_x_context_", tag, "simple"), w = 9, h = 6.5)
  # (c) fitted slope (or pred effect on the slope) along target informativity, one line per context level
  L_c <- do.call(rbind, lapply(seq_len(nrow(grid_c)), function(i) with(grid_c[i, ],
            L_at(pv, LANGS[[language]], POSS[[position]],
                 list(informativity = informativity, context_informativity = context_informativity)))))
  post <- B %*% t(L_c)
  out$slopes <- grid_c %>%
    mutate(est = colMeans(post), l95 = apply(post, 2, quantile, 0.025), u95 = apply(post, 2, quantile, 0.975),
           position = factor(position, names(POSS)),
           context_lab = factor(context_informativity, c(-1, 0, 1), c("low (-1 SD)", "mean", "high (+1 SD)")))
  ylab_c <- if (is.null(pred)) "coarticulation slope (95% CrI)" else
    paste0(pred, " effect on the coarticulation slope\n(per 1 SD ", pred, ", 95% CrI)")
  g <- ggplot(out$slopes, aes(informativity, est, colour = context_lab, fill = context_lab)) +
    { if (!is.null(pred)) geom_hline(yintercept = 0, linetype = 2, colour = "grey50") } +
    geom_ribbon(aes(ymin = l95, ymax = u95), alpha = 0.12, colour = NA) +
    geom_line(linewidth = 0.9) + facet_grid(position ~ language) +
    labs(x = "target informativity (SD)", y = ylab_c, title = "brms",
         colour = "context informativity", fill = "context informativity") + theme_b
  save_b(g, paste0("target_x_context_", tag, "slopes"), w = 9, h = 6.5)
  invisible(out)
}
tc_slope_b   <- tc_block_b()            # = cell 12
tc_unigram_b <- tc_block_b("unigram")   # = cell 13, unigram
tc_gain_b    <- tc_block_b("gain")      # = cell 13, gain

## Posterior coarticulation slope by language × position × informativity

In [ ]:
# ---- Posterior coarticulation slope by language x position x informativity (brms) ----
# slope = d(target_f0)/d(context_f0) at informativity = -1, 0, +1 SD,
# with unigram, gain, unit_pos, duration and portions at 0 (their means); tone terms cancel.
suppressPackageStartupMessages({ library(posterior); library(dplyr); library(tidyr); library(ggplot2) })

tt_s    <- delete.response(terms(m_lang_pos, fixed.only = TRUE))
lme_nms <- names(fixef(m_lang_pos))
brm_nms <- paste0("b_", ifelse(lme_nms == "(Intercept)", "Intercept", lme_nms))
stopifnot(all(brm_nms %in% variables(fit_b)))

grid <- expand_grid(lang = c(-0.5, 0.5), pos = c(-0.5, 0.5), informativity = c(-1, 0, 1))

# one contrast row per grid cell: X(context_f0 = 1) - X(context_f0 = 0)
slope_row <- function(lang, pos, inf) {
  base <- both[1, ]
  base$lang <- lang; base$pos <- pos; base$informativity <- inf
  for (v in setdiff(all.vars(tt_s), c("context_f0", "lang", "pos", "informativity"))) base[[v]] <- 0
  x1 <- model.matrix(tt_s, transform(base, context_f0 = 1))[, lme_nms, drop = FALSE]
  x0 <- model.matrix(tt_s, transform(base, context_f0 = 0))[, lme_nms, drop = FALSE]
  x1 - x0
}
L_slope <- do.call(rbind, Map(slope_row, grid$lang, grid$pos, grid$informativity))

B <- unclass(as_draws_matrix(fit_b, variable = brm_nms))[, brm_nms]
slope_draws <- B %*% t(L_slope)                     # draws x grid cells

ss_b <- grid %>%
  mutate(est = colMeans(slope_draws),
         l95 = apply(slope_draws, 2, quantile, 0.025),
         u95 = apply(slope_draws, 2, quantile, 0.975),
         l80 = apply(slope_draws, 2, quantile, 0.10),
         u80 = apply(slope_draws, 2, quantile, 0.90),
         language = if_else(lang > 0, "Southern Min", "Mandarin"),
         position = if_else(pos > 0, "carryover", "anticipatory"))
ss_b %>% select(language, position, informativity, est, l95, u95) %>%
  mutate(across(c(est, l95, u95), ~ round(.x, 3))) %>% print(n = Inf)

# informativity effect on the slope (high - low, 2 SD), with its own posterior interval
hi <- which(grid$informativity == 1); lo <- which(grid$informativity == -1)
d_hl <- slope_draws[, hi] - slope_draws[, lo]
grid[hi, c("lang", "pos")] %>%
  mutate(language = if_else(lang > 0, "Southern Min", "Mandarin"),
         position = if_else(pos > 0, "carryover", "anticipatory"),
         high_minus_low = round(colMeans(d_hl), 3),
         l95 = round(apply(d_hl, 2, quantile, 0.025), 3),
         u95 = round(apply(d_hl, 2, quantile, 0.975), 3),
         p_neg = round(colMeans(d_hl < 0), 3)) %>%
  select(language, position, high_minus_low, l95, u95, p_neg) %>% print()

# ---- Plot (same layout as the lmer version) ----
dodge <- position_dodge(width = 0.3)
p_slopes_b <- ggplot(ss_b, aes(factor(informativity, c(-1, 0, 1), c("low", "mean", "high")), est,
                              colour = language, group = language)) +
  geom_linerange(aes(ymin = l95, ymax = u95), position = dodge) +
  geom_linerange(aes(ymin = l80, ymax = u80), linewidth = 1.4, position = dodge) +
  geom_point(size = 2.5, position = dodge) +
  geom_line(position = dodge) +
  facet_wrap(~ position) +
  labs(x = "informativity (±1 SD)",
       y = "coarticulation slope\n(posterior mean, 80% / 95% CrI)", colour = "language") +
  theme_minimal(base_size = 12) + theme(legend.position = "bottom")
p_slopes_b
ggsave(file.path(FIT_DIR, paste0("slopes_by_informativity_brms_", INF_SCALE, WSUF, ".png")),
       p_slopes_b, width = 10, height = 4.5, dpi = 300)

### Exploratory: the matched-informativity comparison from the brms posterior (run after the brms analysis cells)

In [ ]:
st <- both %>% group_by(lang_lab) %>% summarise(m = mean(informativity_raw), s = sd(informativity_raw))
m <- setNames(st$m, st$lang_lab); s <- setNames(st$s, st$lang_lab); sd_p <- sd(both$informativity_raw)
L_lang <- function(lg, lv, C) L_at(c("unigram", "informativity"), lv, 0.5,
                                   list(context_informativity = (C - m[[lg]]) / s[[lg]])) * sd_p / s[[lg]]
stopifnot(exists("dens0"), exists("fmt_dir"), exists("PRIOR_B_SD"))   # from the figure-coefficients cell

options(width = 250)
for (C in c(0.08, mean(both$informativity_raw))) {
  L <- rbind(TM = L_lang("TM", -0.5, C), TSM = L_lang("TSM", 0.5, C),
             "TSM - TM" = L_lang("TSM", 0.5, C) - L_lang("TM", -0.5, C))
  post <- B %*% t(L)
  res <- bind_rows(lapply(seq_len(nrow(L)), function(j) {
    x <- post[, j]; p_neg <- mean(x < 0); p_pos <- mean(x > 0)
    prior_sd <- PRIOR_B_SD * sqrt(sum(L[j, ]^2))
    tibble(context_raw = C, row = rownames(L)[j],
           estimate = mean(x), l95 = quantile(x, .025), u95 = quantile(x, .975),
           pd = max(p_neg, p_pos),
           BF10 = dnorm(0, 0, prior_sd) / dens0(x),
           direction = ifelse(p_neg >= p_pos, "< 0", "> 0"),
           BF_dir = ifelse(min(p_neg, p_pos) == 0, Inf, max(p_neg, p_pos) / min(p_neg, p_pos)),
           BF_dir_cap = length(x) - 1)
  })) %>%
    mutate(across(c(estimate, l95, u95), ~ round(.x, 3)), pd = round(pd, 3),
           BF01 = bf_with_label(1 / BF10), BF10 = bf_with_label(BF10),
           BF_dir = fmt_dir(BF_dir, BF_dir_cap)) %>%
    select(-BF_dir_cap)
  print(as.data.frame(res), row.names = FALSE)
}

## Paper figures and the coefficients behind them

In [ ]:
# ===================== Paper figures from the brms model =====================
# Population-level predictions from the posterior draws of fit_b (posterior mean, 80% and 95% credible bands).
# Every predictor not shown is held at 0 = its mean (tone dummies at 0, so tone effects average out);
# context informativity is held at each language's own mean, except in figure 4 (common raw value).
suppressPackageStartupMessages({ library(posterior); library(dplyr); library(tidyr); library(ggplot2) })
stopifnot(INF_SCALE == "within")   # figures 3–4 convert within-language z back to bits

lme_nms <- names(fixef(m_lang_pos))
brm_nms <- paste0("b_", ifelse(lme_nms == "(Intercept)", "Intercept", lme_nms))
B       <- unclass(as_draws_matrix(fit_b, variable = brm_nms))[, brm_nms, drop = FALSE]
tt_p    <- delete.response(terms(m_lang_pos, fixed.only = TRUE))
ctr_p   <- attr(model.matrix(m_lang_pos), "contrasts")

LANG <- c(Mandarin = -0.5, "Southern Min" = 0.5)
POS  <- c(anticipatory = -0.5, carryover = 0.5)
LANG_COL <- c(Mandarin = "#0072B2", "Southern Min" = "#D55E00")   # Okabe–Ito; CVD-checked
C_REF <- 0.08                                                     # figure 4: common context informativity (bits)

# design row at lang / pos, all other numeric predictors at 0 unless set in `at`
X_row <- function(lang, pos, at = list()) {
  nd <- both[1, ]
  for (v in setdiff(all.vars(tt_p), c("lang", "pos"))) if (is.numeric(nd[[v]])) nd[[v]] <- 0
  nd$lang <- lang; nd$pos <- pos
  for (v in names(at)) nd[[v]] <- at[[v]]
  model.matrix(tt_p, nd, contrasts.arg = ctr_p)[, lme_nms, drop = FALSE]
}
# coarticulation slope d(target_f0)/d(context_f0)
slope_row <- function(lang, pos, at = list())
  X_row(lang, pos, modifyList(at, list(context_f0 = 1))) - X_row(lang, pos, modifyList(at, list(context_f0 = 0)))
# unigram effect on the slope, d²(target_f0)/d(context_f0)d(unigram)
uni_row <- function(lang, pos, at = list())
  slope_row(lang, pos, modifyList(at, list(unigram = 1))) - slope_row(lang, pos, modifyList(at, list(unigram = 0)))

post_summ <- function(L, grid) {
  post <- B %*% t(L)
  grid %>% mutate(est = colMeans(post),
                  l95 = apply(post, 2, quantile, 0.025), u95 = apply(post, 2, quantile, 0.975),
                  l80 = apply(post, 2, quantile, 0.10),  u80 = apply(post, 2, quantile, 0.90),
                  position = factor(position, names(POS)))
}
rows_for <- function(grid, f) do.call(rbind, lapply(seq_len(nrow(grid)), function(i) f(grid[i, ])))

# language-specific mean / SD (bits) of target informativity, and its central 95% range
inf_st <- both %>%
  mutate(language = if_else(lang_lab == "TSM", "Southern Min", "Mandarin")) %>%
  group_by(language) %>%
  summarise(m = mean(informativity_raw), s = sd(informativity_raw),
            lo = quantile(informativity_raw, 0.025), hi = quantile(informativity_raw, 0.975))
ST <- split(inf_st, inf_st$language)

theme_fig <- theme_minimal(base_size = 12) +
  theme(legend.position = "bottom", panel.grid.minor = element_blank(),
        axis.title   = element_text(size = 12),
        axis.text    = element_text(size = 10, colour = "grey20"),
        strip.text   = element_text(size = 12, face = "bold"),
        legend.text  = element_text(size = 12),
        legend.title = element_text(size = 12))
pos_lab <- as_labeller(c(anticipatory = "Anticipatory", carryover = "Carryover"))
bands <- list(geom_ribbon(aes(ymin = l95, ymax = u95), alpha = 0.15, colour = NA),
              geom_ribbon(aes(ymin = l80, ymax = u80), alpha = 0.30, colour = NA),
              geom_line(linewidth = 0.9))
lang_scales <- list(scale_colour_manual(values = LANG_COL, name = NULL),
                    scale_fill_manual(values = LANG_COL, name = NULL))
save_fig <- function(g, nm, w = 6.5, h = 3.2) {
  print(g)
  for (ext in c("png", "pdf"))
    ggsave(file.path(FIT_DIR, paste0(nm, "_brms.", ext)), g, width = w, height = h, dpi = 300, bg = "white")
}

# ---- Figure 1: coarticulation (target F0 along context F0) ----
g1 <- expand_grid(language = names(LANG), position = names(POS), context_f0 = seq(-2, 2, by = 0.1))
fig1 <- post_summ(rows_for(g1, function(r) X_row(LANG[[r$language]], POS[[r$position]],
                                                list(context_f0 = r$context_f0))), g1)
p1 <- ggplot(fig1, aes(context_f0, est, colour = language, fill = language)) + bands + lang_scales +
  facet_wrap(~ position, labeller = pos_lab) +
  labs(x = "Context F0 (z)", y = "Predicted target F0 (z)") + theme_fig
save_fig(p1, "fig1_coarticulation")

# ---- Figure 2: word surprisal and the coarticulation slope ----
g2 <- expand_grid(language = names(LANG), position = names(POS), unigram = seq(-2, 2, by = 0.1))
fig2 <- post_summ(rows_for(g2, function(r) slope_row(LANG[[r$language]], POS[[r$position]],
                                                    list(unigram = r$unigram))), g2)
p2 <- ggplot(fig2, aes(unigram, est, colour = language, fill = language)) + bands + lang_scales +
  facet_wrap(~ position, labeller = pos_lab) +
  labs(x = "Word surprisal (SD)", y = "Coarticulation slope\n(target F0 on context F0)") + theme_fig
save_fig(p2, "fig2_word_surprisal")

# ---- Figure 3: target informativity and the coarticulation slope (bits, each language over its own range) ----
g3 <- bind_rows(lapply(names(LANG), function(lg)
  expand_grid(language = lg, position = names(POS),
              bits = seq(ST[[lg]]$lo, ST[[lg]]$hi, length.out = 60))))
fig3 <- post_summ(rows_for(g3, function(r) slope_row(LANG[[r$language]], POS[[r$position]],
            list(informativity = (r$bits - ST[[r$language]]$m) / ST[[r$language]]$s,
                 context_informativity = 0))), g3)
p3 <- ggplot(fig3, aes(bits, est, colour = language, fill = language)) + bands + lang_scales +
  facet_wrap(~ position, labeller = pos_lab) +
  labs(x = "Target tone informativity (bits)", y = "Coarticulation slope\n(target F0 on context F0)") + theme_fig
save_fig(p3, "fig3_informativity")

# ---- Figure 4: Southern Min - Mandarin difference in the word-surprisal effect, along target informativity ----
# at the SAME raw target informativity (bits) and the same raw context informativity (C_REF), over the range
# both languages cover (overlap of the central 95% ranges). The slope of this line = the language difference in
# how target informativity changes the word-surprisal effect on coarticulation.
lo4 <- max(inf_st$lo); hi4 <- min(inf_st$hi)
if (lo4 >= hi4) { warning("no overlap in target informativity; plotting the union instead")
                  lo4 <- min(inf_st$lo); hi4 <- max(inf_st$hi) }
z_at <- function(lg, bits) (bits - ST[[lg]]$m) / ST[[lg]]$s
g4 <- expand_grid(position = names(POS), bits = seq(lo4, hi4, length.out = 60))
fig4 <- post_summ(rows_for(g4, function(r) {
  at_l <- function(lg) list(informativity = z_at(lg, r$bits), context_informativity = z_at(lg, C_REF))
  uni_row(LANG[["Southern Min"]], POS[[r$position]], at_l("Southern Min")) -
    uni_row(LANG[["Mandarin"]],   POS[[r$position]], at_l("Mandarin"))
}), g4)
p4 <- ggplot(fig4, aes(bits, est)) +
  geom_hline(yintercept = 0, linetype = 2, colour = "grey50") +
  geom_ribbon(aes(ymin = l95, ymax = u95), alpha = 0.15, fill = "grey30") +
  geom_ribbon(aes(ymin = l80, ymax = u80), alpha = 0.30, fill = "grey30") +
  geom_line(linewidth = 0.9, colour = "grey15") +
  facet_wrap(~ position, labeller = pos_lab) +
  labs(x = "Target tone informativity (bits)",
       y = "Southern Min − Mandarin:\nword-surprisal effect on the slope") + theme_fig
save_fig(p4, "fig4_lang_diff_surprisal_x_informativity")

In [ ]:
# ===================== Coefficients behind each line of figures 1–4 (brms) =====================
# estimate = posterior mean; 95% CrI; pd = probability of direction;
# BF10 = Savage–Dickey Bayes factor for effect != 0 vs = 0 (prior of the contrast implied by b ~ normal(0, 0.5));
# BF01 = 1 / BF10. Large BFs are capped in printing (">1000").
# direction / BF_dir = favoured sign of the effect and its posterior odds P(favoured sign) / P(other sign),
# i.e. the directional Bayes factor (prior odds 1:1 because the priors are symmetric around 0).
PRIOR_B_SD <- 0.5   # must match prior(normal(0, 0.5), class = "b") in the brms cell

dens0 <- function(x) {                              # posterior density at 0
  if (requireNamespace("logspline", quietly = TRUE))
    return(logspline::dlogspline(0, logspline::logspline(x)))
  d <- density(x, n = 2048)
  if (0 < min(d$x) || 0 > max(d$x)) 0 else approx(d$x, d$y, xout = 0)$y
}
coef_row <- function(figure, line, position, units, L) {
  L <- matrix(L, nrow = 1)
  post <- as.numeric(B %*% t(L))
  prior_sd <- PRIOR_B_SD * sqrt(sum(L^2))
  p_neg <- mean(post < 0); p_pos <- mean(post > 0)
  tibble(figure, line, position, units,
         estimate = mean(post), l95 = quantile(post, 0.025), u95 = quantile(post, 0.975),
         pd = max(p_neg, p_pos),
         BF10 = dnorm(0, 0, prior_sd) / dens0(post),
         direction = ifelse(p_neg >= p_pos, "< 0", "> 0"),
         # if no draw has the other sign, the odds are only known to exceed n_draws - 1
         BF_dir = ifelse(min(p_neg, p_pos) == 0, Inf, max(p_neg, p_pos) / min(p_neg, p_pos)),
         BF_dir_cap = length(post) - 1)
}
# derivative along target informativity (per 1 within-language SD); linear, so evaluated between 0 and 1
d_inf <- function(f) function(lang, pos, at = list())
  f(lang, pos, modifyList(at, list(informativity = 1))) - f(lang, pos, modifyList(at, list(informativity = 0)))
per_bit <- function(lg) 0.01 / ST[[lg]]$s            # per-SD -> per-0.01-bit conversion for language lg

out <- list()
for (p in names(POS)) {
  pv <- POS[[p]]
  # Figure 1: coarticulation slope
  L1 <- lapply(LANG, function(lv) slope_row(lv, pv))
  out <- c(out, list(
    coef_row("1 coarticulation", "Mandarin", p, "slope", L1$Mandarin),
    coef_row("1 coarticulation", "Southern Min", p, "slope", L1$`Southern Min`),
    coef_row("1 coarticulation", "Southern Min - Mandarin", p, "slope", L1$`Southern Min` - L1$Mandarin)))
  # Figure 2: word-surprisal effect on the slope
  L2 <- lapply(LANG, function(lv) uni_row(lv, pv))
  out <- c(out, list(
    coef_row("2 word surprisal", "Mandarin", p, "per 1 SD surprisal", L2$Mandarin),
    coef_row("2 word surprisal", "Southern Min", p, "per 1 SD surprisal", L2$`Southern Min`),
    coef_row("2 word surprisal", "Southern Min - Mandarin", p, "per 1 SD surprisal", L2$`Southern Min` - L2$Mandarin)))
  # Figure 3: target-informativity effect on the slope (context informativity at each language's mean)
  L3 <- lapply(LANG, function(lv) d_inf(slope_row)(lv, pv, list(context_informativity = 0)))
  out <- c(out, list(
    coef_row("3 informativity", "Mandarin", p, "per 1 SD (within)", L3$Mandarin),
    coef_row("3 informativity", "Southern Min", p, "per 1 SD (within)", L3$`Southern Min`),
    coef_row("3 informativity", "Mandarin", p, "per 0.01 bits", L3$Mandarin * per_bit("Mandarin")),
    coef_row("3 informativity", "Southern Min", p, "per 0.01 bits", L3$`Southern Min` * per_bit("Southern Min")),
    coef_row("3 informativity", "Southern Min - Mandarin", p, "per 0.01 bits",
             L3$`Southern Min` * per_bit("Southern Min") - L3$Mandarin * per_bit("Mandarin"))))
  # Figure 4: slope of the difference line = how target informativity changes the word-surprisal effect,
  # per 0.01 bits, at context informativity = C_REF bits in both languages
  L4 <- lapply(names(LANG), function(lg)
    d_inf(uni_row)(LANG[[lg]], pv, list(context_informativity = (C_REF - ST[[lg]]$m) / ST[[lg]]$s)) * per_bit(lg))
  names(L4) <- names(LANG)
  out <- c(out, list(
    coef_row("4 surprisal x informativity", "Mandarin", p, "per 0.01 bits", L4$Mandarin),
    coef_row("4 surprisal x informativity", "Southern Min", p, "per 0.01 bits", L4$`Southern Min`),
    coef_row("4 surprisal x informativity", "Southern Min - Mandarin (line slope)", p, "per 0.01 bits",
             L4$`Southern Min` - L4$Mandarin)))
}
fmt_bf <- function(x) ifelse(!is.finite(x) | x > 1000, ">1000", ifelse(x < 0.001, "<0.001", formatC(x, digits = 3, format = "fg")))
# Lee & Wagenmakers (2013) labels for BF >= 1; "–" when BF < 1 (the evidence favours the other hypothesis,
# whose BF in the other column then carries the label)
lw_label <- function(bf) cut(bf, c(-Inf, 1, 3, 10, 30, 100, Inf), right = FALSE,
                             labels = c("–", "anecdotal", "moderate", "strong", "very strong", "extreme"))
bf_with_label <- function(bf) paste0(fmt_bf(bf), " (", lw_label(bf), ")")
fmt_dir <- function(bf, cap) ifelse(is.finite(bf), paste0(fmt_bf(bf), " (", lw_label(bf), ")"),
                                    paste0(">", cap, " (", lw_label(cap), ")"))

coef_tab <- bind_rows(out) %>%
  mutate(position = factor(position, names(POS), labels = c("a", "c")),
         line = line %>% sub("Southern Min", "TSM", ., fixed = TRUE) %>% sub("Mandarin", "TM", ., fixed = TRUE)) %>%
  arrange(figure, position) %>%
  mutate(across(c(estimate, l95, u95), ~ round(.x, 3)), pd = round(pd, 3),
         BF01 = bf_with_label(1 / BF10), BF10 = bf_with_label(BF10),
         BF_dir = fmt_dir(BF_dir, BF_dir_cap)) %>%
  select(-BF_dir_cap)
options(width = 250)
print(as.data.frame(coef_tab), row.names = FALSE)
write_csv(coef_tab, file.path(FIT_DIR, "figure_coefficients_brms.csv"))

### Exploratory: dedicated carryover model for the matched-informativity comparison (≈ 10 min)

In [ ]:
# ===================== Dedicated model: unigram x target informativity x language =====================
# One position (carryover by default), compared at a COMMON raw context informativity.
#   inf_t : target informativity, centered within language, per 1 pooled SD (raw bits / sd_p)
#   inf_c : context informativity, centered at C_REF bits (common to both languages), per 1 pooled SD
# Then  context_f0:unigram:inf_t:lang  = Southern Min - Mandarin difference in how target informativity
# changes the unigram effect on the coarticulation slope, at context informativity = C_REF.
library(lmerTest)
POSITION <- "carryover"       # "anticipatory" for the comparison position
C_REF    <- 0.08              # bits; inside the overlap of the two languages
FILE_DED <- file.path(FIT_DIR, paste0("m_dedicated_unigram_inf_", POSITION, "_c", C_REF, ".rds"))

sd_p  <- sd(both$informativity_raw)
d_ded <- both %>%
  filter(pos == if (POSITION == "carryover") 0.5 else -0.5) %>%
  group_by(lang_lab) %>%
  mutate(inf_t = (informativity_raw - mean(informativity_raw)) / sd_p) %>%
  ungroup() %>%
  mutate(inf_c = (context_informativity_raw - C_REF) / sd_p,
         tone_pair_l = droplevels(tone_pair_l))
cat(nrow(d_ded), "rows |", POSITION, "| pooled SD =", round(sd_p, 4), "bits\n")
print(d_ded %>% group_by(lang_lab) %>%
        summarise(n = n(), ctx_bits_mean = mean(context_informativity_raw),
                  share_ctx_ge_C = mean(context_informativity_raw >= C_REF)))

f_ded <- as.formula(paste(
  "target_f0 ~ context_f0 * unigram * inf_t * inf_c * lang + context_f0 * gain * lang +", TONE_TERMS,
  "+ duration + target_portion + context_portion + context_f0 * unit_pos * lang +",
  "(1 | onset_segment_l) + (1 | offset_segment_l) + (1 | speaker) +",
  "(1 + context_f0 + context_f0:unigram + context_f0:gain | tone_pair_l)"))

m_ded <- if (file.exists(FILE_DED)) readRDS(FILE_DED) else NULL
if (is.null(m_ded)) {
  t0 <- Sys.time()
  m_ded <- lmer(f_ded, data = d_ded, REML = TRUE, control = lmerControl(optimizer = "bobyqa"))
  cat("fitted in", round(difftime(Sys.time(), t0, units = "mins"), 1), "min\n")
  saveRDS(m_ded, FILE_DED)
}

# ---- key contrasts, by coefficient name (order-independent lookup) ----
b_ded <- fixef(m_ded)
cf <- function(vars) {
  hit <- names(b_ded)[sapply(strsplit(names(b_ded), ":"), function(x) setequal(x, vars))]
  stopifnot(length(hit) == 1); hit
}
# unigram x target-informativity effect on the slope, at lang = lv and context informativity = C bits
key_L <- function(lv, C = C_REF) {
  L <- setNames(numeric(length(b_ded)), names(b_ded))
  k <- (C - C_REF) / sd_p                                     # inf_c value for C bits
  L[cf(c("context_f0", "unigram", "inf_t"))]                  <- 1
  L[cf(c("context_f0", "unigram", "inf_t", "lang"))]          <- lv
  L[cf(c("context_f0", "unigram", "inf_t", "inf_c"))]         <- k
  L[cf(c("context_f0", "unigram", "inf_t", "inf_c", "lang"))] <- k * lv
  L
}
rows <- list(); info <- list()
for (C in c(C_REF, 0.09, 0.10)) {
  rows <- c(rows, list(key_L(-0.5, C), key_L(0.5, C), key_L(0.5, C) - key_L(-0.5, C)))
  info <- c(info, list(tibble(context_bits = C, row = c("Mandarin", "Southern Min", "Southern Min - Mandarin"))))
}
ded_tab <- bind_cols(bind_rows(info),
  contest(as_lmerModLmerTest(m_ded), do.call(rbind, rows), joint = FALSE, ddf = "Satterthwaite", confint = TRUE) %>%
    as.data.frame() %>%
    transmute(estimate = Estimate, se = `Std. Error`, df, t = `t value`, l95 = lower, u95 = upper, p = `Pr(>|t|)`)) %>%
  mutate(across(c(estimate, se, t, l95, u95), ~ round(.x, 3)), df = round(df, 1), p = signif(p, 3))
print(as.data.frame(ded_tab), row.names = FALSE)

# the C_REF difference row should equal this coefficient
tidy_coefs(m_ded) %>% filter(term == cf(c("context_f0", "unigram", "inf_t", "lang"))) %>% fmt() %>% print()